# 지도학습_분류분석_컬럼선택법_클래스불균형처리_모델성능향상시키기

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [4]:
data = pd.read_csv("./data/salary2.csv")

In [5]:
data.head(2)

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       46043 non-null  object
 2   education       48842 non-null  object
 3   education-num   48842 non-null  int64 
 4   marital-status  48842 non-null  object
 5   occupation      46033 non-null  object
 6   relationship    48842 non-null  object
 7   race            48842 non-null  object
 8   sex             48842 non-null  object
 9   capital-gain    48842 non-null  int64 
 10  capital-loss    48842 non-null  int64 
 11  hours-per-week  48842 non-null  int64 
 12  native-country  47985 non-null  object
 13  class           48842 non-null  object
dtypes: int64(5), object(9)
memory usage: 5.2+ MB


# train/test 분리
- 학습용/검증용 데이터 나누기

In [7]:
from sklearn.model_selection import train_test_split

In [8]:
train, test = train_test_split(data, test_size=0.4, random_state=42, stratify=data['class'])
# tratify=data['class'] 중요 -> 클래스 비율 유지, 안 하면 클래스 불균형이 더 심해질 수 있음

In [9]:
train.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
47291,18,Private,Some-college,10,Never-married,Transport-moving,Own-child,White,Male,2907,0,30,United-States,<=50K
25800,31,Private,11th,7,Never-married,Craft-repair,Not-in-family,White,Male,0,2001,40,United-States,<=50K
44902,34,Private,Bachelors,13,Never-married,Other-service,Not-in-family,White,Female,0,0,35,United-States,<=50K
41893,59,Private,11th,7,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,<=50K
16531,41,Private,Prof-school,15,Married-civ-spouse,Prof-specialty,Wife,White,Female,0,0,40,United-States,>50K


In [10]:
test.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
5127,52,Private,Bachelors,13,Divorced,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,<=50K
34071,44,NaN,Bachelors,13,Married-civ-spouse,NaN,Wife,White,Female,0,0,16,United-States,>50K
29790,41,Private,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,<=50K
18182,65,Without-pay,7th-8th,4,Widowed,Farming-fishing,Unmarried,White,Female,0,0,50,United-States,<=50K
42585,57,Private,Some-college,10,Widowed,Tech-support,Not-in-family,White,Female,0,0,16,United-States,<=50K


# 전처리
* 중복제거
* 결측값 삭제
* class 컬럼 label encoding

In [11]:
# 중복 제거
train = train.drop_duplicates()
test = test.drop_duplicates()

In [13]:
# 결측치 제거
train = train.dropna()
test = test.dropna()

In [14]:
# 인덱스 정리
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

In [15]:
train.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,18,Private,Some-college,10,Never-married,Transport-moving,Own-child,White,Male,2907,0,30,United-States,<=50K
1,31,Private,11th,7,Never-married,Craft-repair,Not-in-family,White,Male,0,2001,40,United-States,<=50K
2,34,Private,Bachelors,13,Never-married,Other-service,Not-in-family,White,Female,0,0,35,United-States,<=50K
3,59,Private,11th,7,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,<=50K
4,41,Private,Prof-school,15,Married-civ-spouse,Prof-specialty,Wife,White,Female,0,0,40,United-States,>50K


In [16]:
test.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,52,Private,Bachelors,13,Divorced,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,<=50K
1,41,Private,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,<=50K
2,65,Without-pay,7th-8th,4,Widowed,Farming-fishing,Unmarried,White,Female,0,0,50,United-States,<=50K
3,57,Private,Some-college,10,Widowed,Tech-support,Not-in-family,White,Female,0,0,16,United-States,<=50K
4,24,Private,Bachelors,13,Never-married,Tech-support,Not-in-family,White,Female,3674,0,40,United-States,<=50K


### Label Encoding

**class(종속변수)** 가 예를 들어:

원래값
- '>50K'
- '<=50K'

이렇게 문자형이면 모델이 이해 못함.

그래서:

변환
- '<=50K' → 0
- '>50K' → 1

-> 이건 분류 문제의 **종속변수(정답y)** 인코딩 과정

In [17]:
from sklearn.preprocessing import LabelEncoder

In [18]:
le = LabelEncoder()
le.fit(train['class'])
train['class'] = le.transform(train['class'])
test['class'] = le.transform(test['class'])

In [19]:
train.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,18,Private,Some-college,10,Never-married,Transport-moving,Own-child,White,Male,2907,0,30,United-States,0
1,31,Private,11th,7,Never-married,Craft-repair,Not-in-family,White,Male,0,2001,40,United-States,0
2,34,Private,Bachelors,13,Never-married,Other-service,Not-in-family,White,Female,0,0,35,United-States,0
3,59,Private,11th,7,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,0
4,41,Private,Prof-school,15,Married-civ-spouse,Prof-specialty,Wife,White,Female,0,0,40,United-States,1


In [20]:
test.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,52,Private,Bachelors,13,Divorced,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,0
1,41,Private,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,0
2,65,Without-pay,7th-8th,4,Widowed,Farming-fishing,Unmarried,White,Female,0,0,50,United-States,0
3,57,Private,Some-college,10,Widowed,Tech-support,Not-in-family,White,Female,0,0,16,United-States,0
4,24,Private,Bachelors,13,Never-married,Tech-support,Not-in-family,White,Female,3674,0,40,United-States,0


# 중요 컬럼 선택하기
* EDA를 통해서 종속변수와 중요한 관계가 있는 변수들만 선택
* 수치형 변수는 상관분석 결과를 통해 선택, 범주형 변수는 카이제곱 통계량 분석을 통해 선택
* 머신러닝 알고리즘을 통한 1차 분석 후 중요하게 사용된 변수만 선택

# tree 계열 모델의 feature_importance로 선택하기

In [2]:
# X, y 분리 - 독립변수 / 종속변수 분리
X_train = train.drop('class', axis=1)
y_train = train['class']
display(X_train.head())
y_train.value_counts()

NameError: name 'train' is not defined

In [26]:
X_test = test.drop('class', axis=1)
y_test = test['class']
display(X_test.head())
y_test.value_counts()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,52,Private,Bachelors,13,Divorced,Craft-repair,Not-in-family,White,Male,0,0,40,United-States
1,41,Private,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States
2,65,Without-pay,7th-8th,4,Widowed,Farming-fishing,Unmarried,White,Female,0,0,50,United-States
3,57,Private,Some-college,10,Widowed,Tech-support,Not-in-family,White,Female,0,0,16,United-States
4,24,Private,Bachelors,13,Never-married,Tech-support,Not-in-family,White,Female,3674,0,40,United-States


class
0    12463
1     4201
Name: count, dtype: int64

## One-Hot Encoding

### 원핫인코딩 핵심 요약
1️⃣ 왜 쓰는 거야?

👉 머신러닝 모델은 숫자만 이해함
👉 그런데 문자형 데이터(job, gender 등)는 그대로 못 씀
👉 그래서 숫자로 바꿔야 함

2️⃣ 그냥 숫자로 바꾸면 안 되는 이유
student → 0
engineer → 1
teacher → 2


이렇게 하면 모델이

2 > 1 > 0
teacher가 제일 크네?

라고 착각함 ❌

👉 직업에는 크기 개념이 없음
👉 그래서 이 방법은 위험

3️⃣ 그래서 원핫인코딩을 씀

각 값을 새 컬럼으로 만들어버림

job_student	1 / job_engineer 0 / job_teacher 0

✔ 순서 개념 사라짐
✔ 안전하게 학습 가능

4️⃣ 언제 쓰는 거야?

👉 입력 변수(X)가 문자형이고 순서가 없을 때

예:

직업

성별

결혼 여부

지역

5️⃣ 라벨인코딩이랑 차이

원핫인코딩 - X(입력 변수, 명목형)

라벨인코딩 - y(정답) 또는 순서형 변수 

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

- 원핫인코딩 함수 to_ohe :

1️⃣ 문자 컬럼 찾고
2️⃣ 거기에만 원핫 적용하고
3️⃣ 숫자는 그대로 두고
4️⃣ train/test 둘 다 변환하는 역할

In [1]:
# 원핫인코딩 함수 to_ohe
def to_ohe(X_train, X_test):
    cat_cols = X_train.select_dtypes(include='object').columns
    ct = ColumnTransformer(transformers=[
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
         cat_cols)], remainder='passthrough')
    ct.set_output(transform='pandas')
    X_train = ct.fit_transform(X_train)
    X_test = ct.transform(X_test)
    return X_train, X_test
    

In [27]:
X_train, X_test = to_ohe(X_train, X_test)
display(X_train.head())
display(X_test.head())

,ohe__workclass_ Local-gov,ohe__workclass_ Private,ohe__workclass_ Self-emp-inc,ohe__workclass_ Self-emp-not-inc,ohe__workclass_ State-gov,ohe__workclass_ Without-pay,ohe__education_ 11th,ohe__education_ 12th,ohe__education_ 1st-4th,ohe__education_ 5th-6th,...,ohe__native-country_ Thailand,ohe__native-country_ Trinadad&Tobago,ohe__native-country_ United-States,ohe__native-country_ Vietnam,ohe__native-country_ Yugoslavia,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,18,10,2907,0,30
1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,31,7,0,2001,40
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,34,13,0,0,35
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,59,7,0,0,40
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,41,15,0,0,40


,ohe__workclass_ Local-gov,ohe__workclass_ Private,ohe__workclass_ Self-emp-inc,ohe__workclass_ Self-emp-not-inc,ohe__workclass_ State-gov,ohe__workclass_ Without-pay,ohe__education_ 11th,ohe__education_ 12th,ohe__education_ 1st-4th,ohe__education_ 5th-6th,...,ohe__native-country_ Thailand,ohe__native-country_ Trinadad&Tobago,ohe__native-country_ United-States,ohe__native-country_ Vietnam,ohe__native-country_ Yugoslavia,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,52,13,0,0,40
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,41,9,0,0,40
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,65,4,0,0,50
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,57,10,0,0,16
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,24,13,3674,0,40


# 의사결정나무 분석 후 feature_importance 추출

In [29]:
# 의사결정나무 분류 모델을 가져옴
# 성능 확인용 함수 (accuracy_score → 정확도 계산, classification_report → 정밀도, 재현율, f1-score 출력)
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score, classification_report

dtc = DecisionTreeClassifier(...)
이건 마치

"의사결정나무 기계 하나 생성"

- fit() : dtc.fit(X_train, y_train)

기계한테 데이터랑 정답을 주고
👉 "공부해!" 시키는 것

아직 예측 안 함.
그냥 규칙 배우는 단계.

- predict() : dtc.predict(X_test)

공부한 기계가
👉 새로운 문제(X_test)를 보고
👉 정답을 예측하는 것


### 흐름 
1️⃣ 모델 생성

2️⃣ fit() → 학습

3️⃣ predict() → 예측

4️⃣ 평가


sklearn은 모든 모델을 동일한 방식으로 쓰게 만들었어.
model = 모델()
model.fit()
model.predict()


In [30]:
dtc = DecisionTreeClassifier(max_depth=4, random_state=10) # 모델 틀 만들기
dtc.fit(X_train, y_train) # 모델 학습시키기 : X_train(입력 데이터), y_train(정답)
pred1 = dtc.predict(X_test) # 학습 끝난 모델이 -> X_test를 보고 -> 정답을 예측
print(accuracy_score(y_test, pred1)) # 정확도 계산
print(classification_report(y_test, pred1)) # classification_report (precision recall f1-score support)

0.8380940950552088
              precision    recall  f1-score   support

           0       0.86      0.94      0.90     12463
           1       0.75      0.54      0.63      4201

    accuracy                           0.84     16664
   macro avg       0.80      0.74      0.76     16664
weighted avg       0.83      0.84      0.83     16664



In [35]:
# 각 컬럼이 얼마나 예측에 기여했는지 수치로 보여줌 - 컬럼별 중요도 판단
dtc_importance = pd.DataFrame(dtc.feature_importances_, index=dtc.feature_names_in_, columns=['importance'])
dtc_importance.sort_values(by='importance', ascending=False).head(20)

,importance
ohe__marital-status_ Married-civ-spouse,0.502716
remainder__capital-gain,0.237002
remainder__education-num,0.220531
remainder__capital-loss,0.026382
remainder__hours-per-week,0.011622
remainder__age,0.001748
ohe__workclass_ Private,0.000000
ohe__education_ 12th,0.000000
ohe__education_ 1st-4th,0.000000
ohe__education_ 5th-6th,0.000000


# 의사결정나무에서 중요하게 본 변수만 선택해서 분석

In [39]:
dtc_train = train[['age', 'education-num', 'marital-status', 'capital-gain', 'capital-loss', 'hours-per-week', 'class']]

In [40]:
dtc_train

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week,class
0,18,10,Never-married,2907,0,30,0
1,31,7,Never-married,0,2001,40,0
2,34,13,Never-married,0,0,35,0
3,59,7,Married-civ-spouse,0,0,40,0
4,41,15,Married-civ-spouse,0,0,40,1
...,...,...,...,...,...,...,...
24505,66,4,Divorced,0,0,40,0
24506,48,13,Married-civ-spouse,0,0,40,0
24507,42,10,Divorced,0,0,40,0
24508,47,14,Never-married,0,0,35,1


In [42]:
dtc_train.columns

Index(['age', 'education-num', 'marital-status', 'capital-gain',
       'capital-loss', 'hours-per-week', 'class'],
      dtype='object')

In [43]:
dtc_test = test[dtc_train.columns]
dtc_test

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week,class
0,52,13,Divorced,0,0,40,0
1,41,9,Married-civ-spouse,0,0,40,0
2,65,4,Widowed,0,0,50,0
3,57,10,Widowed,0,0,16,0
4,24,13,Never-married,3674,0,40,0
...,...,...,...,...,...,...,...
16659,33,9,Divorced,0,0,20,0
16660,29,13,Never-married,4650,0,40,0
16661,51,13,Married-civ-spouse,0,1672,70,0
16662,34,13,Married-civ-spouse,7298,0,60,1


In [45]:
dtc_X_train = dtc_train.drop('class', axis=1)
dtc_y_train = dtc_train['class']
dtc_X_test = dtc_test.drop('class', axis=1)
dtc_y_test = dtc_test['class']

In [46]:
dtc_X_train.head()

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week
0,18,10,Never-married,2907,0,30
1,31,7,Never-married,0,2001,40
2,34,13,Never-married,0,0,35
3,59,7,Married-civ-spouse,0,0,40
4,41,15,Married-civ-spouse,0,0,40


In [49]:
dtc_y_train.head()

0    0
1    0
2    0
3    0
4    1
Name: class, dtype: int64

In [47]:
dtc_X_test.head()

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week
0,52,13,Divorced,0,0,40
1,41,9,Married-civ-spouse,0,0,40
2,65,4,Widowed,0,0,50
3,57,10,Widowed,0,0,16
4,24,13,Never-married,3674,0,40


In [50]:
dtc_y_test.head()

0    0
1    0
2    0
3    0
4    0
Name: class, dtype: int64

In [52]:
dtc_X_train, dtc_X_test = to_ohe(dtc_X_train, dtc_X_test)
display(dtc_X_train)
display(dtc_X_test)

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,1.0,0.0,0.0,18,10,2907,0,30
1,0.0,0.0,0.0,1.0,0.0,0.0,31,7,0,2001,40
2,0.0,0.0,0.0,1.0,0.0,0.0,34,13,0,0,35
3,0.0,1.0,0.0,0.0,0.0,0.0,59,7,0,0,40
4,0.0,1.0,0.0,0.0,0.0,0.0,41,15,0,0,40
...,...,...,...,...,...,...,...,...,...,...,...
24505,0.0,0.0,0.0,0.0,0.0,0.0,66,4,0,0,40
24506,0.0,1.0,0.0,0.0,0.0,0.0,48,13,0,0,40
24507,0.0,0.0,0.0,0.0,0.0,0.0,42,10,0,0,40
24508,0.0,0.0,0.0,1.0,0.0,0.0,47,14,0,0,35


,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,0.0,0.0,0.0,52,13,0,0,40
1,0.0,1.0,0.0,0.0,0.0,0.0,41,9,0,0,40
2,0.0,0.0,0.0,0.0,0.0,1.0,65,4,0,0,50
3,0.0,0.0,0.0,0.0,0.0,1.0,57,10,0,0,16
4,0.0,0.0,0.0,1.0,0.0,0.0,24,13,3674,0,40
...,...,...,...,...,...,...,...,...,...,...,...
16659,0.0,0.0,0.0,0.0,0.0,0.0,33,9,0,0,20
16660,0.0,0.0,0.0,1.0,0.0,0.0,29,13,4650,0,40
16661,0.0,1.0,0.0,0.0,0.0,0.0,51,13,0,1672,70
16662,0.0,1.0,0.0,0.0,0.0,0.0,34,13,7298,0,60


In [53]:
dtc2 = DecisionTreeClassifier(max_depth=4, random_state=10)
dtc2.fit(dtc_X_train, dtc_y_train)
pred2 = dtc2.predict(dtc_X_test)
print(accuracy_score(dtc_y_test, pred2))
print(classification_report(dtc_y_test, pred2))

0.8380940950552088
              precision    recall  f1-score   support

           0       0.86      0.94      0.90     12463
           1       0.75      0.54      0.63      4201

    accuracy                           0.84     16664
   macro avg       0.80      0.74      0.76     16664
weighted avg       0.83      0.84      0.83     16664



# randomforest의 경우
 - randomforest : Decision Tree 여러 개 만들어서 다수결로 결정하는 모델

In [54]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [56]:
rfc = RandomForestClassifier(max_depth=4, n_jobs=-1,random_state=10)
rfc.fit(X_train, y_train)
pred1 = rfc.predict(X_test)
print(accuracy_score(y_test, pred1))
print(classification_report(y_test, pred1))

0.8127700432069132
              precision    recall  f1-score   support

           0       0.80      1.00      0.89     12463
           1       0.95      0.27      0.42      4201

    accuracy                           0.81     16664
   macro avg       0.88      0.63      0.65     16664
weighted avg       0.84      0.81      0.77     16664



In [59]:
rfc_importance = pd.DataFrame(rfc.feature_importances_, index=rfc.feature_names_in_, columns=['importance'])
rfc_importance.sort_values(by='importance', ascending=False).head(50)

,importance
ohe__marital-status_ Married-civ-spouse,0.228391
remainder__capital-gain,0.184174
remainder__education-num,0.114874
ohe__marital-status_ Never-married,0.086539
remainder__age,0.069498
remainder__capital-loss,0.041603
ohe__relationship_ Not-in-family,0.035654
ohe__occupation_ Exec-managerial,0.031593
ohe__sex_ Male,0.030942
ohe__relationship_ Own-child,0.029844


# XGBOOST의 경우
- 부스팅(Boosting) : 트리를 하나 만들고 -> 틀린 부분을 다음 트리가 보완 -> 또 틀린 부분을 다음 트리가 보완

- 계속 “틀린 것”을 집중 공략함.

In [60]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [61]:
xgb = XGBClassifier(max_depth=4, n_jobs=-1,random_state=10)
xgb.fit(X_train, y_train)
pred2 = xgb.predict(X_test)
print(accuracy_score(y_test, pred2))
print(classification_report(y_test, pred2))

0.8701992318771004
              precision    recall  f1-score   support

           0       0.89      0.94      0.92     12463
           1       0.78      0.67      0.72      4201

    accuracy                           0.87     16664
   macro avg       0.84      0.80      0.82     16664
weighted avg       0.87      0.87      0.87     16664



In [62]:
xgb_importance = pd.DataFrame(xgb.feature_importances_, index=xgb.feature_names_in_, columns=['importance'])
xgb_importance.sort_values(by='importance', ascending=False).head(50)

,importance
ohe__marital-status_ Married-civ-spouse,0.383847
remainder__education-num,0.084336
remainder__capital-gain,0.048743
ohe__occupation_ Exec-managerial,0.045218
ohe__occupation_ Other-service,0.041056
ohe__occupation_ Farming-fishing,0.029075
remainder__capital-loss,0.026672
ohe__relationship_ Own-child,0.020889
ohe__occupation_ Handlers-cleaners,0.018728
remainder__age,0.015096


그 파일에서

Decision Tree

Random Forest

XGBoost

이 3개를 왜 다 돌렸냐면…

👉 "어떤 모델이 salary2에 제일 잘 맞는지 비교하려고"

근데 너는 AutoML 쓸 거잖아?

AutoGluon 은

✔ 이 3개 모델 다 돌림
✔ 하이퍼파라미터 튜닝 자동
✔ 앙상블까지 자동

그래서 너가 직접 비교 안 해도 됨.

💡 그럼 왜 배웠냐?

AutoML이 내부에서 하는 일을 이해하려고.

AutoML은
Decision Tree도 써보고
Random Forest도 써보고
XGBoost도 써보고
제일 잘 맞는 걸 고르는 기계

이걸 이해하려고 그 예시 3개를 다 보여준 거야.

# 클래스 불균형을 처리하는 방법


## 1️⃣ 클래스 불균형이란?

클래스 불균형은 **데이터에서 한쪽 종류의 값이 너무 많고, 다른 쪽은 너무 적은 상태**를 말한다.  
예를 들어 “연봉이 5천만 원 이상인 사람(1)”과 “그 이하인 사람(0)”을 예측할 때  
- 0(이하) 사람이 100명  
- 1(이상) 사람이 20명  

이라면, 데이터가 **0이 훨씬 많고 1이 적은 불균형한 상태**이다.

이럴 때는 모델이 대부분 “0이라고만 예측해도” 정확도가 높아 보여서 착각하기 쉽다.  
하지만 진짜 중요한 건 “1을 얼마나 잘 맞추느냐”이다.

---

## 2️⃣ 왜 문제가 될까?

모델은 보통 많은 데이터를 더 믿는다.  
그래서 다수(예: 0) 쪽을 너무 잘 배우고, 소수(예: 1) 쪽을 무시하게 된다.  

결국 **희귀하지만 중요한 데이터**(예: 사기 거래, 암 진단 등)를 못 알아보는 일이 생긴다.  
이건 “시험문제 대부분은 쉬운 문제인데, 어려운 문제를 다 틀리는 학생”과 비슷하다.

## 1️⃣ 클래스 불균형이 뭐냐?

salary2가 예를 들어

0 (저연봉) = 75%

1 (고연봉) = 25%

이런 식이면 → 불균형 데이터

❗ 문제점

모델이 전부 0이라고 예측해도
정확도 75% 나옴 😑

근데 우리는 고연봉(1)을 잘 맞추는 게 중요하잖아?

그래서

Accuracy만 보면 안 된다

이게 불균형 파트의 출발점이야.

## 🎯 2️⃣ 그래서 평가지표를 바꾼다

불균형에서는

❌ Accuracy

✅ Recall

✅ F1-score

✅ ROC-AUC

을 본다.

특히

👉 1을 얼마나 잘 맞추는지 (Recall)

이게 핵심.

## 3️⃣ 해결 방법 1: 데이터 나눌 때 주의하기
### 데이터는 그대로, 모델만 조정

데이터를 **훈련(train)** 과 **시험(test)** 으로 나눌 때,  
각 부분에도 **클래스 비율이 비슷하게** 들어가야 한다.

train/test 나눌 때 -> 0/1 비율 유지
이거 안 하면 test에 1이 거의 없을 수도 있음.

이럴 때 **`stratify` 옵션** 을 사용한다.  
이건 “비율 맞춰 나누기”라고 생각하면 된다.
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, stratify=y, random_state=10)
```

## 4️⃣ 해결 방법 2: 모델에게 ‘가중치’를 주기

소수 클래스(적은 데이터 쪽)에 **더 중요한 점수(weight)** 를 주면 된다.  
이건 “시험에서 어려운 문제를 맞추면 점수를 더 주는 것”과 비슷하다.

- 예: `class_weight='balanced'`  
  이렇게 하면 모델이 자동으로 “적은 데이터 쪽에 점수”를 더 준다.

이 방법은 **데이터를 새로 만들지 않고**, 모델만 살짝 조정하는 방법이다.
* decisionTree 모델의 경우 class_weight="balanced" 이진/다중분류에 사용가능
* random_forest의 경우 class_weight="balanced" 이진/다중분류에 사용가능
* xgboost 의 경우 scale_pos_weight= 0 / 1 비율. 단, 이진분류(0,1)인 경우
  * ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
* lightgbm 의 경우 is_unbalance=True. 단, 이진분류(0,1)인 경우

# DecisionTree의 경우
* class_weight="balanced"

In [64]:
# class_weight='balanced' 써서 데이터 불균형 문제 해결

dtc2 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=10)
dtc2.fit(dtc_X_train, dtc_y_train)
pred2 = dtc2.predict(dtc_X_test)
print(accuracy_score(dtc_y_test, pred2))
print(classification_report(dtc_y_test, pred2))

0.7623019683149304
              precision    recall  f1-score   support

           0       0.94      0.73      0.82     12463
           1       0.52      0.87      0.65      4201

    accuracy                           0.76     16664
   macro avg       0.73      0.80      0.73     16664
weighted avg       0.83      0.76      0.78     16664



# RandomForest의 경우
* class_weight="balanced"

In [66]:
rfc = RandomForestClassifier(max_depth=4, class_weight="balanced", n_jobs=-1,random_state=10)
rfc.fit(X_train, y_train)
pred1 = rfc.predict(X_test)
print(accuracy_score(y_test, pred1))
print(classification_report(y_test, pred1))

0.75
              precision    recall  f1-score   support

           0       0.95      0.70      0.81     12463
           1       0.50      0.89      0.64      4201

    accuracy                           0.75     16664
   macro avg       0.73      0.80      0.72     16664
weighted avg       0.84      0.75      0.77     16664



# XGBoost의 경우
* scale_pos_weight= 1/0의 비율
* ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

In [69]:
y_train.value_counts()

class
0    18325
1     6185
Name: count, dtype: int64

In [67]:
ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

In [68]:
ratio

np.float64(2.9628132578819724)

In [71]:
xgb = XGBClassifier(max_depth=4, n_jobs=-1,random_state=10)
xgb.fit(X_train, y_train)
pred2 = xgb.predict(X_test)
print(accuracy_score(y_test, pred2))
print(classification_report(y_test, pred2))

0.8701992318771004
              precision    recall  f1-score   support

           0       0.89      0.94      0.92     12463
           1       0.78      0.67      0.72      4201

    accuracy                           0.87     16664
   macro avg       0.84      0.80      0.82     16664
weighted avg       0.87      0.87      0.87     16664



In [1]:
# recall 올라가고 precision, accuracy 내려감

xgb = XGBClassifier(max_depth=4, scale_pos_weight=ratio, n_jobs=-1,random_state=10)
xgb.fit(X_train, y_train)
pred2 = xgb.predict(X_test)
print(accuracy_score(y_test, pred2))
print(classification_report(y_test, pred2))

NameError: name 'XGBClassifier' is not defined

In [73]:
from lightgbm import LGBMClassifier

In [74]:
lgbm = LGBMClassifier(max_depth=3, n_estimators=500, n_jobs=-1, random_state=10, verbose=-1)
lgbm.fit(X_train, y_train)
pred = lgbm.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.89      0.94      0.92     12463
           1       0.79      0.67      0.72      4201

    accuracy                           0.87     16664
   macro avg       0.84      0.80      0.82     16664
weighted avg       0.87      0.87      0.87     16664



In [76]:
# is_unbalance=True
lgbm = LGBMClassifier(max_depth=3, is_unbalance=True, n_estimators=500, n_jobs=-1, random_state=10, verbose=-1)
lgbm.fit(X_train, y_train)
pred = lgbm.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.95      0.82      0.88     12463
           1       0.62      0.87      0.72      4201

    accuracy                           0.83     16664
   macro avg       0.78      0.84      0.80     16664
weighted avg       0.87      0.83      0.84     16664



In [77]:
lgbm = LGBMClassifier(max_depth=3, n_estimators=500, n_jobs=-1, random_state=10, verbose=-1)
lgbm.fit(X_train, y_train)
pred = lgbm.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.89      0.94      0.92     12463
           1       0.79      0.67      0.72      4201

    accuracy                           0.87     16664
   macro avg       0.84      0.80      0.82     16664
weighted avg       0.87      0.87      0.87     16664



## 5️⃣ 해결 방법 3: 데이터 양을 맞춰주기(데이터증폭/축소)

* imbalanced-learn, smote/cluster-centroids
* https://imbalanced-learn.org/stable/
* 머신러닝 알고리즘을 통해 비슷한 데이터를 생성/축소
* **반드시 train 데이터에만 적용**

In [79]:
# !pip install imbalanced-learn

### (1) 소수 데이터를 늘리기 (Over-sampling)
오버샘플링(SMOTE): 소수 클래스를 “비슷한 점들 사이”에서 보간해 새 데이터를 생성한다.<br> 경계에서 과적합이 생기지 않도록 k_neighbors를 3~10에서 조정해본다.<br>

소수 데이터를 **복사하거나 새로 만들어서** 비율을 맞춘다.<br>

- 단순 복사: 같은 데이터를 여러 번 사용한다.  
- **SMOTE** 방법: 비슷한 데이터를 살짝 바꿔서 새로운 데이터를 만든다.  
  → 예를 들어, 친구 A와 B의 키가 160cm, 165cm라면  
  → 그 중간값 162cm 근처의 새 데이터를 만드는 느낌이다.

* 숫자와 범주형 데이터가 섞여있으므로 SMOTENC 로 train데이터만 증폭

In [81]:
from imblearn.over_sampling import SMOTENC

In [84]:
dtc_train

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week,class
0,18,10,Never-married,2907,0,30,0
1,31,7,Never-married,0,2001,40,0
2,34,13,Never-married,0,0,35,0
3,59,7,Married-civ-spouse,0,0,40,0
4,41,15,Married-civ-spouse,0,0,40,1
...,...,...,...,...,...,...,...
24505,66,4,Divorced,0,0,40,0
24506,48,13,Married-civ-spouse,0,0,40,0
24507,42,10,Divorced,0,0,40,0
24508,47,14,Never-married,0,0,35,1


In [98]:
dtc_test

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week,class
0,52,13,Divorced,0,0,40,0
1,41,9,Married-civ-spouse,0,0,40,0
2,65,4,Widowed,0,0,50,0
3,57,10,Widowed,0,0,16,0
4,24,13,Never-married,3674,0,40,0
...,...,...,...,...,...,...,...
16659,33,9,Divorced,0,0,20,0
16660,29,13,Never-married,4650,0,40,0
16661,51,13,Married-civ-spouse,0,1672,70,0
16662,34,13,Married-civ-spouse,7298,0,60,1


In [86]:
dtc_X_train2 = dtc_train.drop('class', axis=1)
dtc_y_train2 = dtc_train['class']

In [94]:
dtc_X_test2 = dtc_test.drop('class', axis=1)
dtc_y_test2 = dtc_test['class']

In [87]:
dtc_X_train2

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week
0,18,10,Never-married,2907,0,30
1,31,7,Never-married,0,2001,40
2,34,13,Never-married,0,0,35
3,59,7,Married-civ-spouse,0,0,40
4,41,15,Married-civ-spouse,0,0,40
...,...,...,...,...,...,...
24505,66,4,Divorced,0,0,40
24506,48,13,Married-civ-spouse,0,0,40
24507,42,10,Divorced,0,0,40
24508,47,14,Never-married,0,0,35


In [89]:
dtc_y_train2.value_counts() 
# 데이터 개수 다름

class
0    18325
1     6185
Name: count, dtype: int64

In [90]:
smtnc = SMOTENC(categorical_features=[2],  k_neighbors=5, random_state=10)
smt_X_train, smt_y_train = smtnc.fit_resample(dtc_X_train2, dtc_y_train2)
display(smt_X_train)
smt_y_train.value_counts()
# 데이터 개수 증폭시킴 -> 0,1 데이터 개수 같아짐

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week
0,18,10,Never-married,2907,0,30
1,31,7,Never-married,0,2001,40
2,34,13,Never-married,0,0,35
3,59,7,Married-civ-spouse,0,0,40
4,41,15,Married-civ-spouse,0,0,40
...,...,...,...,...,...,...
36645,40,12,Married-civ-spouse,15024,0,50
36646,35,9,Married-civ-spouse,0,0,60
36647,46,9,Divorced,0,0,38
36648,49,9,Married-civ-spouse,0,0,40


class
0    18325
1    18325
Name: count, dtype: int64

In [99]:
dtc_X_test2

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,0.0,0.0,0.0,52,13,0,0,40
1,0.0,1.0,0.0,0.0,0.0,0.0,41,9,0,0,40
2,0.0,0.0,0.0,0.0,0.0,1.0,65,4,0,0,50
3,0.0,0.0,0.0,0.0,0.0,1.0,57,10,0,0,16
4,0.0,0.0,0.0,1.0,0.0,0.0,24,13,3674,0,40
...,...,...,...,...,...,...,...,...,...,...,...
16659,0.0,0.0,0.0,0.0,0.0,0.0,33,9,0,0,20
16660,0.0,0.0,0.0,1.0,0.0,0.0,29,13,4650,0,40
16661,0.0,1.0,0.0,0.0,0.0,0.0,51,13,0,1672,70
16662,0.0,1.0,0.0,0.0,0.0,0.0,34,13,7298,0,60


In [95]:
smt_X_train, dtc_X_test2 = to_ohe(smt_X_train, dtc_X_test2)

In [100]:
smt_X_train

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,1.0,0.0,0.0,18,10,2907,0,30
1,0.0,0.0,0.0,1.0,0.0,0.0,31,7,0,2001,40
2,0.0,0.0,0.0,1.0,0.0,0.0,34,13,0,0,35
3,0.0,1.0,0.0,0.0,0.0,0.0,59,7,0,0,40
4,0.0,1.0,0.0,0.0,0.0,0.0,41,15,0,0,40
...,...,...,...,...,...,...,...,...,...,...,...
36645,0.0,1.0,0.0,0.0,0.0,0.0,40,12,15024,0,50
36646,0.0,1.0,0.0,0.0,0.0,0.0,35,9,0,0,60
36647,0.0,0.0,0.0,0.0,0.0,0.0,46,9,0,0,38
36648,0.0,1.0,0.0,0.0,0.0,0.0,49,9,0,0,40


In [101]:
dtc_X_test2

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,0.0,0.0,0.0,52,13,0,0,40
1,0.0,1.0,0.0,0.0,0.0,0.0,41,9,0,0,40
2,0.0,0.0,0.0,0.0,0.0,1.0,65,4,0,0,50
3,0.0,0.0,0.0,0.0,0.0,1.0,57,10,0,0,16
4,0.0,0.0,0.0,1.0,0.0,0.0,24,13,3674,0,40
...,...,...,...,...,...,...,...,...,...,...,...
16659,0.0,0.0,0.0,0.0,0.0,0.0,33,9,0,0,20
16660,0.0,0.0,0.0,1.0,0.0,0.0,29,13,4650,0,40
16661,0.0,1.0,0.0,0.0,0.0,0.0,51,13,0,1672,70
16662,0.0,1.0,0.0,0.0,0.0,0.0,34,13,7298,0,60


In [102]:
lgbm = LGBMClassifier(max_depth=3, n_estimators=500, n_jobs=-1, random_state=10, verbose=-1)
lgbm.fit(smt_X_train, smt_y_train)
pred = lgbm.predict(dtc_X_test2)
print(classification_report(dtc_y_test2, pred))

              precision    recall  f1-score   support

           0       0.95      0.80      0.87     12463
           1       0.59      0.87      0.71      4201

    accuracy                           0.82     16664
   macro avg       0.77      0.84      0.79     16664
weighted avg       0.86      0.82      0.83     16664



### (2) 다수 데이터를 줄이기 (Under-sampling)
언더샘플링: 다수 클래스를 일부만 사용한다. 학습은 빨라지지만 정보 손실이 생길 수 있다.<br>
데이터가 너무 많은 쪽(예: 0)을 **일부만 사용**해서 비율을 맞춘다. <br>
하지만 정보가 줄어들 수 있다는 단점이 있다.

In [104]:
dtc_X_train3 = dtc_train.drop('class', axis=1)
dtc_y_train3 = dtc_train['class']

In [107]:
dtc_y_train3.value_counts()

class
0    18325
1     6185
Name: count, dtype: int64

In [103]:
dtc_X_test3 = dtc_test.drop('class', axis=1)
dtc_y_test3 = dtc_test['class']

In [105]:
from imblearn.under_sampling import RandomUnderSampler

In [106]:
rus = RandomUnderSampler(random_state=10)
rus_X, rus_y = rus.fit_resample(dtc_X_train3, dtc_y_train3)
display(rus_X)
rus_y.value_counts()

,age,education-num,marital-status,capital-gain,capital-loss,hours-per-week
4425,43,10,Married-civ-spouse,0,0,40
22532,67,10,Widowed,0,0,25
177,47,9,Married-civ-spouse,3908,0,40
16975,21,9,Never-married,0,0,40
10153,62,6,Married-civ-spouse,0,0,40
...,...,...,...,...,...,...
24491,47,13,Married-civ-spouse,15024,0,60
24499,39,13,Married-civ-spouse,0,1887,40
24500,44,14,Never-married,0,0,40
24501,59,14,Married-civ-spouse,0,0,45


class
0    6185
1    6185
Name: count, dtype: int64

In [108]:
rus_X_train, dtc_X_test3 = to_ohe(rus_X, dtc_X_test3)
display(rus_X_train)
display(dtc_X_test3)

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
4425,0.0,1.0,0.0,0.0,0.0,0.0,43,10,0,0,40
22532,0.0,0.0,0.0,0.0,0.0,1.0,67,10,0,0,25
177,0.0,1.0,0.0,0.0,0.0,0.0,47,9,3908,0,40
16975,0.0,0.0,0.0,1.0,0.0,0.0,21,9,0,0,40
10153,0.0,1.0,0.0,0.0,0.0,0.0,62,6,0,0,40
...,...,...,...,...,...,...,...,...,...,...,...
24491,0.0,1.0,0.0,0.0,0.0,0.0,47,13,15024,0,60
24499,0.0,1.0,0.0,0.0,0.0,0.0,39,13,0,1887,40
24500,0.0,0.0,0.0,1.0,0.0,0.0,44,14,0,0,40
24501,0.0,1.0,0.0,0.0,0.0,0.0,59,14,0,0,45


,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,0.0,0.0,0.0,52,13,0,0,40
1,0.0,1.0,0.0,0.0,0.0,0.0,41,9,0,0,40
2,0.0,0.0,0.0,0.0,0.0,1.0,65,4,0,0,50
3,0.0,0.0,0.0,0.0,0.0,1.0,57,10,0,0,16
4,0.0,0.0,0.0,1.0,0.0,0.0,24,13,3674,0,40
...,...,...,...,...,...,...,...,...,...,...,...
16659,0.0,0.0,0.0,0.0,0.0,0.0,33,9,0,0,20
16660,0.0,0.0,0.0,1.0,0.0,0.0,29,13,4650,0,40
16661,0.0,1.0,0.0,0.0,0.0,0.0,51,13,0,1672,70
16662,0.0,1.0,0.0,0.0,0.0,0.0,34,13,7298,0,60


In [109]:
lgbm = LGBMClassifier(max_depth=3, n_estimators=500, n_jobs=-1, random_state=10, verbose=-1)
lgbm.fit(rus_X_train, rus_y)
pred = lgbm.predict(dtc_X_test3)
print(classification_report(dtc_y_test3, pred))

              precision    recall  f1-score   support

           0       0.95      0.80      0.87     12463
           1       0.60      0.87      0.71      4201

    accuracy                           0.82     16664
   macro avg       0.77      0.84      0.79     16664
weighted avg       0.86      0.82      0.83     16664



# 데이터 증폭과 축소 동시에 하기
* 많은 쪽은 조금 줄이고, 적은 쪽은 조금 늘리는 방법도 있다.<br>  
* 이건 “균형 맞추기” 느낌이다.<br>
* 혼합(SMOTEENN): 오버+언더를 함께 써서 노이즈를 줄이고 균형을 잡는다.<br>

* 비교 지표는 불균형 문제에서는 보통 F1/Recall 위주로 보는 것이 좋다.

In [110]:
dtc_X_train4 = dtc_train.drop('class', axis=1)
dtc_y_train4 = dtc_train['class']

In [111]:
dtc_y_train4.value_counts()

class
0    18325
1     6185
Name: count, dtype: int64

In [112]:
dtc_X_test4 = dtc_test.drop('class', axis=1)
dtc_y_test4 = dtc_test['class']

In [115]:
dtc_X_train4, dtc_X_test4 = to_ohe(dtc_X_train4, dtc_X_test4)

In [116]:
dtc_X_train4.head()

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,1.0,0.0,0.0,18,10,2907,0,30
1,0.0,0.0,0.0,1.0,0.0,0.0,31,7,0,2001,40
2,0.0,0.0,0.0,1.0,0.0,0.0,34,13,0,0,35
3,0.0,1.0,0.0,0.0,0.0,0.0,59,7,0,0,40
4,0.0,1.0,0.0,0.0,0.0,0.0,41,15,0,0,40


In [117]:
dtc_X_test4.head()

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.0,0.0,0.0,0.0,0.0,52,13,0,0,40
1,0.0,1.0,0.0,0.0,0.0,0.0,41,9,0,0,40
2,0.0,0.0,0.0,0.0,0.0,1.0,65,4,0,0,50
3,0.0,0.0,0.0,0.0,0.0,1.0,57,10,0,0,16
4,0.0,0.0,0.0,1.0,0.0,0.0,24,13,3674,0,40


In [113]:
from imblearn.combine import SMOTEENN

In [118]:
smoteenn = SMOTEENN(random_state=10, n_jobs=-1)
sen_X, sen_y = smoteenn.fit_resample(dtc_X_train4, dtc_y_train4)
display(sen_X)
sen_y.value_counts()

,ohe__marital-status_ Married-AF-spouse,ohe__marital-status_ Married-civ-spouse,ohe__marital-status_ Married-spouse-absent,ohe__marital-status_ Never-married,ohe__marital-status_ Separated,ohe__marital-status_ Widowed,remainder__age,remainder__education-num,remainder__capital-gain,remainder__capital-loss,remainder__hours-per-week
0,0.0,0.000000,0.0,1.000000,0.0,0.0,18,10,2907,0,30
1,0.0,0.000000,0.0,1.000000,0.0,0.0,31,7,0,2001,40
2,0.0,0.000000,0.0,1.000000,0.0,0.0,34,13,0,0,35
3,0.0,1.000000,0.0,0.000000,0.0,0.0,40,9,0,0,40
4,0.0,0.000000,0.0,1.000000,0.0,0.0,30,13,0,0,55
...,...,...,...,...,...,...,...,...,...,...,...
24003,0.0,0.341792,0.0,0.658208,0.0,0.0,33,9,0,0,40
24004,0.0,1.000000,0.0,0.000000,0.0,0.0,49,13,7298,0,40
24005,0.0,0.764337,0.0,0.235663,0.0,0.0,50,14,0,0,50
24006,0.0,1.000000,0.0,0.000000,0.0,0.0,40,12,15024,0,50


class
0    13079
1    10929
Name: count, dtype: int64

In [119]:
lgbm = LGBMClassifier(max_depth=3, n_estimators=500, n_jobs=-1, random_state=10, verbose=-1)
lgbm.fit(sen_X, sen_y)
pred = lgbm.predict(dtc_X_test4)
print(classification_report(dtc_y_test4, pred))

              precision    recall  f1-score   support

           0       0.92      0.85      0.89     12463
           1       0.64      0.79      0.71      4201

    accuracy                           0.84     16664
   macro avg       0.78      0.82      0.80     16664
weighted avg       0.85      0.84      0.84     16664

